# NARS demo
One question through the whole pipeline, with every step printed: the LLM translates (facts → triples → hop plan → rules), code checks the rules, ONA reasons, the rule sets vote, and the answer is scored with the three metrics. Run cell 1 once, then edit and run cell 2.

In [ ]:
# ── NARS demo: one question, every step printed (run this cell once, then call demo(...) below) ───────
import json
import os
import re
import unicodedata
from collections import Counter

import nbformat
from openai import OpenAI

import ONA

ONA_PATH = "/Users/mina/Developer/OpenNARS-for-Applications/NAR"
NARS_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"  # translates facts -> triples -> plan -> rules
JUDGE_MODEL = "Qwen/Qwen2.5-72B-Instruct"               # third metric, only asked when lenient fails
BASELINE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"  # an LLM answering directly, for contrast
from keys import deepinfra_key
API_KEY = deepinfra_key()  # $DEEPINFRA_API_KEY or the git-ignored .deepinfra_key
client = OpenAI(api_key=API_KEY, base_url="https://api.deepinfra.com/v1/openai", max_retries=2, timeout=90)


def normalize(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9]", "", re.sub(r"\b(a|an|the)\b", " ", text))


def scores(question, gold, answer):
    """The three metrics of the main notebook: exact, lenient (one contains the other), judge (LLM, if lenient fails)."""
    a, g = normalize(answer or ""), normalize(gold)
    short, long_ = sorted((a, g), key=len)
    result = {"exact": bool(a) and a == g, "lenient": bool(short) and short in long_ and 2 * len(short) >= len(long_)}
    result["judge"] = result["lenient"]
    if answer and not result["lenient"]:
        reply = client.chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=3, messages=[{
            "role": "user", "content": f"Question: {question}\nGold answer: {gold}\nModel answer: {answer}\n"
            "Is the model answer the same entity, date, number or place as the gold answer (not a related or vaguer "
            "one)? Reply with only yes or no."}])
        result["judge"] = (reply.choices[0].message.content or "").strip().lower().startswith("yes")
    return result


def header(n, title, who):
    print(f"\n{'═' * 100}\nSTEP {n}. {title}   [{who}]\n{'═' * 100}")


def show_triples(triples):
    for t in triples:
        note = "   (reverse, added by code)" if t.get("inverse") else \
            f"   (general form of {t['merged_from']}, added by code)" if t.get("merged_from") else ""
        print(f"   {t['subject']} | {ONA.relation_name(t['relation'])} | {t['object']}{note}")


def demo(facts, question, gold, plan=True, votes=3, merge_relations=True):
    """Run NARS end to end on one question and print every step: the LLM's outputs, the code's checks and repairs,
    each voted rule set with its Narsese and ONA output, the vote, the three metrics, and an LLM baseline."""
    ona = ONA.OnaReasoner(client, model=NARS_MODEL, plan=plan, votes=votes, merge_relations=merge_relations)
    answer = ona.solve(facts, question, ONA_PATH)

    header(1, "INPUT", "you")
    print(f"   Question: {question}\n   Gold:     {gold}\n   Facts:")
    for f in facts:
        print(f"     - {f}")

    header(2, "FACTS -> TRIPLES", f"LLM: TRIPLES prompt, {NARS_MODEL.split('/')[-1]}")
    show_triples(ona.last_raw_triples)

    header(3, "ONE NAME PER ENTITY, REVERSE FACTS, GENERAL RELATIONS", "LLM: ALIASES prompt + code")
    merged = {a: c for a, c in ona.aliases.items() if a != c}
    print(f"   Names merged: {merged or 'none'}")
    print("   Triples ONA can use:")
    show_triples(ona.last_triples)

    header(4, "HOP PLAN (from the question only)", "LLM: PLAN prompt")
    print("\n".join(f"   {h}" for h in (ona.plan_text or "(plan off)").splitlines()))

    header(5, f"RULES: up to {votes} rule sets, each checked, run in ONA, then a vote", "LLM: RULES prompt + code + ONA")
    print("   The LLM writes the hop chain as a rule. Code checks every condition against the triples; if one does not\n"
          "   match, the problem goes back to the LLM (a REPAIR turn) or a small automatic fix is made (auto-repair).\n"
          "   Rule set 1 is written at temperature 0, the others at 0.7 (slightly different wording), and the answer\n"
          "   most rule sets agree on wins. Voting stops early once two agree.")
    for k, s in enumerate(ona.samples, 1):
        print(f"\n   ── Rule set {k} {'(temperature 0)' if k == 1 else '(temperature 0.7)'} " + "─" * 60)
        for a, (raw, problems) in enumerate(s["rule_attempts"], 1):
            print(f"   LLM attempt {a}: {' '.join(raw.split())}")
            for p in problems:
                print(f"      code: problem sent back -> {p}")
        print(f"   Auto-repairs by code: {dict(s['auto_repairs']) or 'none'}")
        if not s["rules"]:
            print("   No valid rule: this rule set gives no answer.")
            continue
        for r in s["rules"]:
            chain = "  AND  ".join(f"({c[0]} {c[1]} {c[2]})" for c in r["conditions"])
            print(f"   Checked rule: {chain}  ->  answer is {r['answer']}")
        print("   Narsese sent to ONA:")
        print("\n".join(f"      {line}" for line in s["narsese"].strip().splitlines()))
        lines = [l for l in s["output"].splitlines() if l.startswith("Answer:")]
        print("   ONA output (its answers, with truth values):")
        print("\n".join(f"      {l}" for l in lines) or "      (nothing derived)")
        print(f"   This rule set's answer: {s['answer']}")
    tally = Counter(s["answer"] for s in ona.samples if s["answer"])
    print(f"\n   VOTE: {dict(tally) or 'no answers'}  ->  NARS answer: {answer}")

    header(6, "SCORING", f"code + judge {JUDGE_MODEL.split('/')[-1]}")
    nars_scores = scores(question, gold, answer)
    print(f"   NARS answer: {answer!r}   gold: {gold!r}   ->   {nars_scores}")

    header(7, "LLM BASELINE (same facts, answers directly)", BASELINE_MODEL.split("/")[-1])
    reply = client.chat.completions.create(model=BASELINE_MODEL, temperature=0, max_tokens=64, messages=[{
        "role": "user", "content": "Answer the question using only the facts below. Return only the short answer, "
        "without explanation.\n\nFacts:\n" + "\n".join(f"- {f}" for f in facts) + f"\n\nQuestion: {question}\nAnswer:"}])
    base = (reply.choices[0].message.content or "").strip().split("\n")[0]
    print(f"   Baseline answer: {base!r}   ->   {scores(question, gold, base)}")

    header(8, "SUMMARY", "")
    calls = Counter()
    for call in ona.request_log:
        last = call["messages"][-1]["content"]
        calls[next((n for n, s in [("TRIPLE_REPAIR", "These names"), ("REPAIR", "Those rules"), ("ALIASES", "Group the"),
                                   ("PLAN", "Break the question"), ("RULES", "Write rules"), ("TRIPLES", "Convert the")]
                    if last.startswith(s)), "other")] += 1
    print(f"   LLM calls by NARS: {dict(calls)}   tokens in/out: {ona.usage['input_tokens']}/{ona.usage['output_tokens']}")
    print(f"   The LLM only translated (facts -> triples -> plan -> rules); ONA did the reasoning.")
    return answer


In [ ]:
# ── Try it: change the facts, question and gold, then run this cell ─────────────────────────────────────
facts = [
    "Ann directed Film X.",
    "Ann was born in Paris.",
]
question = "Where was the director of Film X born?"
gold = "Paris"

demo(facts, question, gold)